# FraudShield - Task 2: IEEE-CIS Fraud Detection Dataset Inspection

Welcome to the **FraudShield** exploratory data analysis workspace.

### Dataset Overview
The **IEEE-CIS Fraud Detection** dataset (provided by Vesta Corporation, the world's leading payment service company) is a benchmark real-world e-commerce fraud dataset. It represents real-world transactions with a wide variety of features ranging from device signatures to transaction amounts and behavioral history.

### Objectives of this Notebook:
1. **Load the CSV files** (`train_transaction.csv` and `train_identity.csv`) using Pandas with memory optimization.
2. **Display shapes** and explain the relationship between transactions and identity records.
3. **Display sample rows** across both tables.
4. **Identify numerical and categorical columns**.
5. **Examine missing values** and feature sparsity.
6. **Analyse the target variable** (`isFraud`).
7. **Visualize fraud vs. legitimate transaction distributions** across amount, time, and categories.
8. **Identify potential data quality issues** (extreme sparsity, right-skewed amounts, collinearity, time leaks).
9. **Explain key columns**, including `TransactionDT`, `TransactionID`, and grouped features (`C`, `D`, `M`, `V`, `id`).

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in Python sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import load_config
from src.data.loader import reduce_mem_usage, get_default_paths

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 50)
print("Libraries imported and environment initialized successfully!")

## 1. Load the CSV Files Using Pandas

The dataset is divided into two primary relational tables:
- **`train_transaction.csv`**: Contains payment transaction metadata, card info, address codes, and Vesta-engineered features (394 columns, ~683 MB).
- **`train_identity.csv`**: Contains digital identity and device metadata (e.g. browser, OS, screen resolution, IP proxy) for transactions performed via digital checkouts (41 columns, ~26 MB).

> **Industry Practice - Memory Optimization**:
> Loading large CSVs with standard 64-bit precision consumes >1.8 GB RAM. Using our modular `reduce_mem_usage()` downcasts 64-bit types to 32/16-bit safely, halving memory consumption.

In [ ]:
paths = get_default_paths()
print(f"Transaction File: {paths['transaction']}")
print(f"Identity File:    {paths['identity']}")

# Load datasets (adjust nrows=None to load the complete 590k records)
# Using nrows=100000 for fast interactive exploration, or set to None for the entire dataset
N_ROWS = 100000  # Change to None to load all 590,540 rows

print(f"Loading transaction data (nrows={N_ROWS})...")
df_transaction = pd.read_csv(paths["transaction"], nrows=N_ROWS)
df_transaction = reduce_mem_usage(df_transaction, verbose=True)

print(f"Loading identity data (nrows={N_ROWS})...")
df_identity = pd.read_csv(paths["identity"], nrows=N_ROWS)
df_identity = reduce_mem_usage(df_identity, verbose=True)

print("Datasets loaded successfully!")

## 2. Display Dataset Shapes & Relational Linkage

Let's inspect the dimensionality of both tables and explain how they connect.

In [ ]:
print("=" * 60)
print(f"Transaction Table Shape : {df_transaction.shape[0]:,} rows × {df_transaction.shape[1]} columns")
print(f"Identity Table Shape    : {df_identity.shape[0]:,} rows × {df_identity.shape[1]} columns")
print("=" * 60)

# Check overlap of TransactionID between both tables
common_ids = set(df_transaction["TransactionID"]).intersection(set(df_identity["TransactionID"]))
print(f"Common TransactionIDs in this sample: {len(common_ids):,}")
print(f"Percentage of transactions with identity info: {len(common_ids) / len(df_transaction) * 100:.2f}%")

### Key Observation on Shapes:
- Notice that `train_transaction` has significantly more rows than `train_identity`.
- In the full dataset, there are **590,540 transactions**, but only **144,233 identity rows** (~$24.4\%$).
- **Why?** Not all transactions collect device info (e.g. POS in-person terminal transactions or checkouts without device fingerprinting do not produce identity records). A relational `LEFT JOIN` on `TransactionID` will retain all transactions while generating nulls for transactions lacking identity logs.

## 3. Display Sample Rows

Let's look at the first 5 records of each table to understand their structure and feature values.

In [ ]:
print("--- Transaction Data: First 5 Rows (Selected Key Columns) ---")
key_tx_cols = ["TransactionID", "isFraud", "TransactionDT", "TransactionAmt", "ProductCD", "card1", "card4", "card6", "P_emaildomain"]
df_transaction[key_tx_cols].head()

In [ ]:
print("--- Identity Data: First 5 Rows (Selected Key Columns) ---")
key_id_cols = ["TransactionID", "id_01", "id_02", "id_12", "id_15", "id_28", "id_31", "DeviceType", "DeviceInfo"]
df_identity[key_id_cols].head()

## 4. Identify Numerical and Categorical Columns

Understanding data types is vital because:
1. **Numerical features** require distribution analysis, scaling, and outlier handling.
2. **Categorical features** require encoding (Target encoding, One-Hot encoding, or Ordinal/Label encoding for tree models).
3. **Pseudo-numerical features**: Features like `card1` - `card6`, `addr1`, `addr2` are stored as numbers, but are conceptually categorical identifier codes (e.g. issuing bank code, zip code region).

In [ ]:
# Transaction table dtypes
tx_num = df_transaction.select_dtypes(include=["number"]).columns.tolist()
tx_cat = df_transaction.select_dtypes(exclude=["number"]).columns.tolist()

# Identity table dtypes
id_num = df_identity.select_dtypes(include=["number"]).columns.tolist()
id_cat = df_identity.select_dtypes(exclude=["number"]).columns.tolist()

print("=" * 60)
print(f"Transaction Table: {len(tx_num)} Numerical columns | {len(tx_cat)} Categorical columns")
print(f"Categorical columns in Transaction: {tx_cat}")
print("-" * 60)
print(f"Identity Table   : {len(id_num)} Numerical columns | {len(id_cat)} Categorical columns")
print(f"Categorical columns in Identity: {id_cat}")
print("=" * 60)

## 5. Examine Missing Values and Feature Sparsity

Financial fraud datasets frequently exhibit severe missingness due to optional form fields, heterogeneous payment gateways, and varying levels of device telemetry.

In [ ]:
# Missing values analysis for transactions
tx_missing = df_transaction.isnull().mean() * 100
cols_over_50_pct = (tx_missing > 50).sum()
cols_over_75_pct = (tx_missing > 75).sum()
cols_over_90_pct = (tx_missing > 90).sum()

print(f"Total columns in Transaction: {len(df_transaction.columns)}")
print(f"Columns with > 50% missing values: {cols_over_50_pct} ({cols_over_50_pct / len(df_transaction.columns) * 100:.1f}%)")
print(f"Columns with > 75% missing values: {cols_over_75_pct} ({cols_over_75_pct / len(df_transaction.columns) * 100:.1f}%)")
print(f"Columns with > 90% missing values: {cols_over_90_pct} ({cols_over_90_pct / len(df_transaction.columns) * 100:.1f}%)")

# Visualize missing value distribution
plt.figure(figsize=(10, 4))
sns.histplot(tx_missing, bins=25, kde=True, color="#e67e22")
plt.title("Distribution of Missing Value Percentages Across Transaction Columns", fontsize=12, fontweight="bold")
plt.xlabel("Percentage Missing (%)")
plt.ylabel("Number of Columns")
plt.axvline(50, color="red", linestyle="--", label="50% Missing Threshold")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Analyse the Target Variable: `isFraud`

In fraud detection, the target variable is binary:
- `0`: Legitimate Transaction
- `1`: Fraudulent Transaction

In [ ]:
fraud_counts = df_transaction["isFraud"].value_counts()
fraud_pcts = df_transaction["isFraud"].value_counts(normalize=True) * 100
imbalance_ratio = fraud_counts[0] / fraud_counts[1]

print("=" * 60)
print("TARGET VARIABLE (isFraud) BREAKDOWN:")
print(f"  Legitimate (0) : {fraud_counts[0]:,} transactions ({fraud_pcts[0]:.2f}%)")
print(f"  Fraudulent (1) : {fraud_counts[1]:,} transactions ({fraud_pcts[1]:.2f}%)")
print(f"  Imbalance Ratio: {imbalance_ratio:.1f} legitimate transactions for every 1 fraud")
print("=" * 60)

## 7. Visualize Fraud versus Legitimate Transaction Distribution

In [ ]:
# Visual 1: Target Class Imbalance (Linear and Log Scale)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(ax=axes[0], x="isFraud", data=df_transaction, hue="isFraud", palette=["#2ecc71", "#e74c3c"], legend=False)
axes[0].set_title("Class Distribution (Standard Count Scale)", fontweight="bold")
axes[0].set_xticklabels(["Legitimate (0)", "Fraud (1)"])
axes[0].set_ylabel("Count")

sns.countplot(ax=axes[1], x="isFraud", data=df_transaction, hue="isFraud", palette=["#2ecc71", "#e74c3c"], legend=False)
axes[1].set_yscale("log")
axes[1].set_title("Class Distribution (Log Scale)", fontweight="bold")
axes[1].set_xticklabels(["Legitimate (0)", "Fraud (1)"])
axes[1].set_ylabel("Count (Log Scale)")

plt.tight_layout()
plt.show()

In [ ]:
# Visual 2: Transaction Amounts - Legitimate vs Fraud
plt.figure(figsize=(12, 5))

# Filter out extreme values for cleaner visualization
sns.boxplot(x="isFraud", y="TransactionAmt", data=df_transaction, hue="isFraud", palette=["#2ecc71", "#e74c3c"], showfliers=False, legend=False)
plt.title("Transaction Amount Distribution by Class (Excluding Extreme Outliers)", fontsize=12, fontweight="bold")
plt.xticks([0, 1], ["Legitimate (0)", "Fraud (1)"])
plt.ylabel("Transaction Amount ($)")
plt.tight_layout()
plt.show()

print("Transaction Amount Summary Statistics:")
print(df_transaction.groupby("isFraud")["TransactionAmt"].describe()[["count", "mean", "std", "min", "50%", "max"]])

In [ ]:
# Visual 3: Temporal Distribution across Days
# TransactionDT is in seconds; let's convert to day numbers
df_transaction["Day"] = df_transaction["TransactionDT"] // (24 * 3600)
daily_fraud = df_transaction.groupby("Day")["isFraud"].agg(["count", "sum", "mean"])
daily_fraud.columns = ["Total_Transactions", "Fraud_Count", "Fraud_Rate"]

fig, ax1 = plt.subplots(figsize=(12, 4))
ax2 = ax1.twinx()

ax1.bar(daily_fraud.index, daily_fraud["Total_Transactions"], color="#3498db", alpha=0.5, label="Total Transactions")
ax2.plot(daily_fraud.index, daily_fraud["Fraud_Rate"] * 100, color="#e74c3c", linewidth=2.5, marker="o", label="Fraud Rate (%)")

ax1.set_xlabel("Timeline (Days from Reference Point)")
ax1.set_ylabel("Total Transactions", color="#2980b9")
ax2.set_ylabel("Fraud Rate (%)", color="#c0392b")
plt.title("Transaction Volume and Daily Fraud Rate Over Time", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Visual 4: Product Code (ProductCD) Fraud Distribution
prod_summary = df_transaction.groupby("ProductCD")["isFraud"].agg(["count", "mean"]).reset_index()
prod_summary["Fraud_Rate_Pct"] = prod_summary["mean"] * 100
prod_summary = prod_summary.sort_values(by="Fraud_Rate_Pct", ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(x="ProductCD", y="Fraud_Rate_Pct", data=prod_summary, palette="magma", hue="ProductCD", legend=False)
plt.title("Fraud Rate by Product Code (ProductCD)", fontsize=12, fontweight="bold")
plt.ylabel("Fraud Rate (%)")
plt.xlabel("Product Code")
for idx, row in prod_summary.iterrows():
    plt.text(row.name, row["Fraud_Rate_Pct"] + 0.3, f"{row['Fraud_Rate_Pct']:.1f}%\n(N={int(row['count']):,})", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

## 8. Identify Potential Data Quality Issues

From our exploratory inspection, we identify four critical data quality and modeling challenges:

1. **Severe Sparsity (> 50% Missing Columns)**:
   - Over 200 columns contain more than 50% missing values. Traditional models (like basic Logistic Regression or standard Neural Networks) cannot handle `NaN` natively without deliberate imputation strategies or indicator masks.

2. **Extreme Right-Skew in `TransactionAmt`**:
   - Most transactions are under $100, but max values exceed thousands of dollars. Distance-based models require log transformation or robust outlier-resistant scaling.

3. **High Cardinality in Categorical Columns**:
   - Columns such as `P_emaildomain`, `R_emaildomain`, `DeviceInfo`, and `id_31` (browser version) contain hundreds of unique, noisy strings. Direct one-hot encoding would explode the feature dimension.

4. **Time-Series Nature & Lookahead Risk (`TransactionDT`)**:
   - Transactions are chronologically ordered. Applying standard random $K$-Fold cross-validation would cause **data leakage / look-ahead bias** (training on future transactions to predict past transactions). We must use **Time-based or TimeSeriesSplit validation**.

## 9. Comprehensive Feature Dictionary & Column Explanation

| Feature Group | Column Names | Description & Significance |
| :--- | :--- | :--- |
| **Primary Identifiers** | `TransactionID` | Unique numeric identifier for each transaction; acts as the primary join key with `train_identity.csv`. |
| **Target** | `isFraud` | Binary indicator: `1` for fraudulent transactions, `0` for legitimate. |
| **Time Delta** | `TransactionDT` | Monotonically increasing timedelta in seconds from an undisclosed reference date. (e.g., $86,400 = 24$ hours). Spans ~182 days (approx. 6 months). |
| **Payment Amount** | `TransactionAmt` | Transaction payment amount in USD. Shows right-skewness and distinct decimal patterns (e.g. cents vs whole dollar amounts). |
| **Product Code** | `ProductCD` | Categorical product category code for each order (e.g., W, C, H, R, S). Notably, codes 'C' and 'S' exhibit elevated fraud rates. |
| **Payment Cards** | `card1` – `card6` | Payment card information, including issuer bank identification number (BIN), card brand (`visa`, `mastercard`), card type (`credit`, `debit`), and country. |
| **Addresses** | `addr1`, `addr2` | Purchaser billing region (`addr1`) and billing country code (`addr2`). |
| **Distances** | `dist1`, `dist2` | Geographic distances (e.g., distance between billing address and shipping address or IP address location). |
| **Email Domains** | `P_emaildomain`, `R_emaildomain` | Purchaser and recipient email domain names (e.g. `gmail.com`, `anonymous.com`, `yahoo.com`). Discrepancies often correlate with fraud. |
| **Counting Features** | `C1` – `C14` | Quantitative count metrics (e.g., how many cards are associated with an address, IP frequency count, etc.). Masked for proprietary privacy. |
| **Timedeltas** | `D1` – `D15` | Timedeltas in days (e.g. days since previous transaction of the same card or account). |
| **Match Flags** | `M1` – `M9` | Boolean match indicators (e.g. name on card matches shipping address, billing matches cardholder name). True/False flags. |
| **Vesta Rich Features** | `V1` – `V339` | Engineered features engineered by Vesta (ranking, counts, behavioral flags). High dimensionality and redundancy. |
| **Digital Identity** | `id_01` – `id_38` | Network and identity telemetry (IP risk score, proxy usage, login status, device specs). |
| **Device Attributes** | `DeviceType`, `DeviceInfo` | Hardware and client metadata (e.g. `mobile`, `desktop`, `iOS`, `Windows`, `SM-G960U`). |

## 10. Conclusion and Next Steps for Week 2

### Summary of Findings:
1. **Severe Imbalance**: 96.5% legitimate vs 3.5% fraud. Metric strategy must center on **PR-AUC, Recall, and Precision**, not raw Accuracy.
2. **Data Sparsity**: Over half of the columns contain >50% nulls. Identity metadata is present in only ~24.4% of transactions.
3. **Time Dimension**: `TransactionDT` represents an ongoing timeline, necessitating time-aware train/validation splitting.

### Ready for Week 2!
In Week 2, we will build our **Logistic Regression baseline**, implement a time-stratified validation strategy, and establish benchmark performance metrics.